# Odds Direction Model — EDA Phase 1

**Goal**: Discover which features predict the *direction* of ML model probability movement over the next 12 balls (2 overs).

**Primary Target**: `ml_prob[i+12] - ml_prob[i]` → binary UP/DOWN  
**Comparison Target**: `resource_win_prob[i+12] - resource_win_prob[i]` (physics baseline)  
**Residual Target** (pro): `ml_delta - momentum_baseline` (unexpected movement only)

**Data**: IPL (~1,146 matches, ~273K balls) + PSL (~75K balls)  
**Model**: Global T20 (`models/t20_male_v2/champion_model.joblib`) generates ML probs

---

## Structure
1. **Load & Prepare** — Load training data, generate ML predictions, reconstruct match boundaries
2. **Target Distribution** — ML prob delta vs resource_win_prob delta
3. **Candidate Features** — Engineer momentum/trend features
4. **Correlation Analysis** — Which features correlate with ML prob direction?
5. **Feature Importance** — Quick XGBoost to rank features
6. **SHAP Analysis** — Understand nonlinear effects & interactions
7. **Segment Analysis** — Does importance vary by innings/phase?
8. **Baseline Models** — Naive, momentum, logistic vs XGBoost
9. **Residual Analysis** — Can we predict *unexpected* movement?
10. **Conclusions** — Is direction predictable? What features matter most?

> **Why ML prob over resource_win_prob?**
> - ML encodes venue, pressure, player quality, context — resource_win_prob is just DLS physics
> - Predicting ML movement = directly useful for your betting system
> - Closer to market behavior (market reacts more like ML than DLS)
> 
> **Risk**: Self-referential learning (ODM learns ML's internal patterns, not real edge).  
> **Mitigation**: Momentum baseline comparison — if ODM ≈ momentum, it's useless.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

ROOT = Path('../../')  # workspace root from specs/010-odds-direction-model/
HORIZON = 12  # balls ahead (2 overs)

# TOP_FEATURES used by the XGBLogRegEnsemble model
TOP_FEATURES = [
    'expected_final_score', 'resource_win_prob', 'score_vs_par', 
    'dls_pressure_index', 'projected_vs_venue_avg', 'projected_score',
    'is_powerplay', 'score_per_wicket', 'run_rate_diff', 'required_run_rate',
    'chase_difficulty', 'wickets_times_balls', 'pressure_index', 
    'team_strength_diff', 'rrr_times_wickets', 'overs_remaining',
    'batting_team_win_rate', 'bowling_team_win_rate', 'batting_team_situation_wr',
    'situation_advantage', 'boundary_pct_last_18', 'bowling_team_situation_wr',
    'runs_last_12', 'runs_last_18', 'wickets_last_12'
]

print('Setup complete')

## 1. Load Data & Generate ML Predictions

Training data has ball-by-ball features but NO explicit `match_id` and NO ML model probability.  
We:
1. Load the global T20 model and generate `ml_prob` for every ball
2. Reconstruct match boundaries from innings transitions (2→1)
3. Construct ML prob delta target within each innings

In [ ]:
# Load IPL + PSL training data
df_ipl = pd.read_parquet(ROOT / 'data/ipl_features_v1/training.parquet')
df_ipl['league'] = 'ipl'
df_psl = pd.read_parquet(ROOT / 'data/psl_features_v1/training.parquet')
df_psl['league'] = 'psl'

df = pd.concat([df_ipl, df_psl], ignore_index=True)
print(f'Combined: {len(df):,} balls, {len(df.columns)} columns')
print(f'IPL: {len(df_ipl):,} | PSL: {len(df_psl):,}')

In [ ]:
# Load global T20 model and generate ML probabilities for every ball
model = joblib.load(ROOT / 'models/t20_male_v2/champion_model.joblib')
print(f'Model: {type(model).__name__}')

# Generate raw ML probabilities (P(batting team wins))
X_model = df[TOP_FEATURES]
ml_probs = model.predict_proba(X_model)[:, 1]
df['ml_prob'] = ml_probs

print(f'ML prob range: {df["ml_prob"].min():.4f} to {df["ml_prob"].max():.4f}')
print(f'ML prob mean:  {df["ml_prob"].mean():.4f}')
print(f'resource_win_prob mean: {df["resource_win_prob"].mean():.4f}')
print(f'\nCorrelation ml_prob vs resource_win_prob: {df["ml_prob"].corr(df["resource_win_prob"]):.4f}')

In [ ]:
# Reconstruct match_id from innings transitions (innings goes 2 → 1 at match boundary)
df['inn_reset'] = (df['innings'] == 1) & (df['innings'].shift(1) == 2)
df['match_id'] = df['inn_reset'].cumsum()
df.drop(columns='inn_reset', inplace=True)

# Convert overs_remaining to ball_number within innings
df['ball_in_innings'] = ((20 - df['overs_remaining']) * 6).round().astype(int)
df['over_number'] = (20 - df['overs_remaining']).astype(int).clip(lower=0, upper=19)

# Sort by match, innings, ball — critical for target construction
df = df.sort_values(['match_id', 'innings', 'ball_in_innings']).reset_index(drop=True)

n_matches = df['match_id'].nunique()
print(f'Reconstructed {n_matches} matches, avg {len(df)/n_matches:.0f} balls/match')

## 2. Target Variable Construction

Three targets computed within each `(match_id, innings)` group:

| Target | Formula | What it captures |
|--------|---------|-----------------|
| **`ml_delta_12`** (PRIMARY) | `ml_prob[i+12] - ml_prob[i]` | ML model intelligence movement |
| `rwp_delta_12` (comparison) | `resource_win_prob[i+12] - resource_win_prob[i]` | DLS physics movement |
| `residual_delta_12` (pro) | `ml_delta_12 - momentum_baseline` | Unexpected / non-trivial movement |

**Momentum baseline** = `ml_prob[i] - ml_prob[i-12]` (simple trend continuation)

In [ ]:
# Compute targets within (match_id, innings) groups
grp = df.groupby(['match_id', 'innings'])

# PRIMARY: ML probability delta
df['ml_prob_future'] = grp['ml_prob'].shift(-HORIZON)
df['ml_delta_12'] = df['ml_prob_future'] - df['ml_prob']

# COMPARISON: resource_win_prob delta  
df['rwp_future'] = grp['resource_win_prob'].shift(-HORIZON)
df['rwp_delta_12'] = df['rwp_future'] - df['resource_win_prob']

# MOMENTUM BASELINE: simple trend continuation
df['ml_prob_past'] = grp['ml_prob'].shift(HORIZON)
df['momentum_baseline'] = df['ml_prob'] - df['ml_prob_past']

# RESIDUAL: unexpected movement = actual delta - momentum
df['residual_delta_12'] = df['ml_delta_12'] - df['momentum_baseline']

# Binary direction (PRIMARY target for classification)
df['direction'] = (df['ml_delta_12'] > 0).astype(int)
df['direction_rwp'] = (df['rwp_delta_12'] > 0).astype(int)
df['direction_residual'] = (df['residual_delta_12'] > 0).astype(int)

# Drop rows without valid target
df_valid = df.dropna(subset=['ml_delta_12', 'momentum_baseline']).copy()

print(f'Valid rows: {len(df_valid):,} / {len(df):,} ({len(df_valid)/len(df)*100:.1f}%)')
print(f'\n--- ML Delta (PRIMARY) ---')
print(f'Direction balance: UP={df_valid["direction"].mean():.3f}, DOWN={1-df_valid["direction"].mean():.3f}')
print(df_valid['ml_delta_12'].describe().round(4).to_string())
print(f'\n--- Resource WP Delta (comparison) ---')
print(f'Direction balance: UP={df_valid["direction_rwp"].mean():.3f}, DOWN={1-df_valid["direction_rwp"].mean():.3f}')
print(f'\n--- Residual Delta (unexpected movement) ---')
print(f'Direction balance: UP={df_valid["direction_residual"].mean():.3f}')
print(f'Correlation(ml_delta, rwp_delta): {df_valid["ml_delta_12"].corr(df_valid["rwp_delta_12"]):.4f}')
print(f'Correlation(ml_delta, momentum):  {df_valid["ml_delta_12"].corr(df_valid["momentum_baseline"]):.4f}')

In [ ]:
# Visualize target distributions: ML delta vs resource WP delta
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: ML prob delta (PRIMARY)
axes[0, 0].hist(df_valid['ml_delta_12'], bins=100, edgecolor='none', alpha=0.7, color='steelblue')
axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title(f'ML Prob Δ12 (n={len(df_valid):,})')
axes[0, 0].set_xlabel('ML Probability Delta')

dir_counts = df_valid['direction'].value_counts()
axes[0, 1].bar(['DOWN (0)', 'UP (1)'], [dir_counts.get(0, 0), dir_counts.get(1, 0)], 
            color=['#e74c3c', '#2ecc71'])
axes[0, 1].set_title('ML Direction Balance')

for inn in [1, 2]:
    subset = df_valid[df_valid['innings'] == inn]['ml_delta_12']
    axes[0, 2].hist(subset, bins=80, alpha=0.5, label=f'Inn {inn} (n={len(subset):,})')
axes[0, 2].axvline(0, color='red', linestyle='--')
axes[0, 2].set_title('ML Delta by Innings')
axes[0, 2].legend()

# Row 2: Comparison — resource WP delta, residual, ML vs RWP scatter
axes[1, 0].hist(df_valid['rwp_delta_12'], bins=100, edgecolor='none', alpha=0.7, color='#e67e22')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1, 0].set_title('Resource WP Δ12 (comparison)')
axes[1, 0].set_xlabel('Resource WP Delta')

axes[1, 1].hist(df_valid['residual_delta_12'], bins=100, edgecolor='none', alpha=0.7, color='#9b59b6')
axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1, 1].set_title('Residual Δ12 (unexpected movement)')
axes[1, 1].set_xlabel('Residual Delta')

# Scatter: ML delta vs RWP delta
sample = df_valid.sample(min(5000, len(df_valid)), random_state=42)
axes[1, 2].scatter(sample['rwp_delta_12'], sample['ml_delta_12'], alpha=0.1, s=3, color='steelblue')
axes[1, 2].plot([-0.5, 0.5], [-0.5, 0.5], 'r--', linewidth=1)
axes[1, 2].set_xlabel('Resource WP Delta')
axes[1, 2].set_ylabel('ML Prob Delta')
axes[1, 2].set_title(f'ML vs RWP Delta (r={df_valid["ml_delta_12"].corr(df_valid["rwp_delta_12"]):.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Delta magnitude by phase
def get_phase(row):
    if row['is_powerplay'] == 1:
        return 'Powerplay'
    elif row['is_death_overs'] == 1:
        return 'Death'
    else:
        return 'Middle'

df_valid['phase'] = df_valid.apply(get_phase, axis=1)

print('=== ML Prob Delta by innings × phase ===')
phase_stats = df_valid.groupby(['innings', 'phase']).agg(
    ml_delta_mean=('ml_delta_12', 'mean'),
    ml_delta_std=('ml_delta_12', 'std'),
    pct_up=('direction', 'mean'),
    count=('direction', 'count')
).round(4)
print(phase_stats.to_string())

print('\n=== Momentum baseline correlation by segment ===')
for (inn, phase), grp_data in df_valid.groupby(['innings', 'phase']):
    r = grp_data['ml_delta_12'].corr(grp_data['momentum_baseline'])
    print(f'  Inn {inn} {phase:10s}: r={r:.4f} (n={len(grp_data):,})')

## 3. Engineer Momentum / Trend Features

Now using **ML probability** for momentum features (not resource_win_prob).  
Key insight: We need features that capture *change over time*, not just *current state*.

> ⚠️ Self-referential risk: Some features here use `ml_prob` history.  
> The momentum baseline comparison (Section 9) tests whether ODM learns anything beyond trivial trend continuation.

In [ ]:
# ---- MOMENTUM FEATURES (using ML prob, not resource_win_prob) ----
grp = df_valid.groupby(['match_id', 'innings'])

# ML probability trend (recent trajectory)
df_valid['ml_prob_delta_last_6'] = df_valid['ml_prob'] - grp['ml_prob'].shift(6)
df_valid['ml_prob_delta_last_12'] = df_valid['ml_prob'] - grp['ml_prob'].shift(12)

# ML prob velocity & acceleration
df_valid['ml_prob_velocity'] = df_valid['ml_prob_delta_last_6'] / 6  # per-ball rate
prev_velocity = grp['ml_prob_delta_last_6'].shift(6) / 6
df_valid['ml_prob_acceleration'] = df_valid['ml_prob_velocity'] - prev_velocity

# Resource WP trend (physics comparison)
df_valid['rwp_delta_last_6'] = df_valid['resource_win_prob'] - grp['resource_win_prob'].shift(6)
df_valid['rwp_delta_last_12'] = df_valid['resource_win_prob'] - grp['resource_win_prob'].shift(12)

# ML-RWP divergence: is ML deviating from physics?
df_valid['ml_rwp_gap'] = df_valid['ml_prob'] - df_valid['resource_win_prob']
df_valid['ml_rwp_gap_delta_6'] = df_valid['ml_rwp_gap'] - grp['ml_rwp_gap'].shift(6).values

# Scoring momentum: compare recent windows
df_valid['runs_last_6'] = df_valid['runs_last_12'] - grp['runs_last_12'].shift(6).fillna(0)
prev_runs_6 = grp['runs_last_12'].shift(6) - grp['runs_last_12'].shift(12)
df_valid['runs_delta_6v6'] = df_valid['runs_last_6'] - prev_runs_6.fillna(0)

# Wicket cluster
df_valid['wickets_delta_6'] = df_valid['wickets_last_12'] - grp['wickets_last_12'].shift(6).fillna(0)

# Phase transition features
df_valid['balls_to_death'] = np.clip((16 * 6) - df_valid['ball_in_innings'], 0, 120)
df_valid['entering_death'] = (df_valid['balls_to_death'] <= 12).astype(int)

# Resource efficiency
df_valid['resource_efficiency'] = df_valid['resource_win_prob'] / df_valid['resource_pct'].clip(lower=0.01)

print(f'Engineered features. NaN counts in new features:')
new_feats = ['ml_prob_delta_last_6', 'ml_prob_delta_last_12', 'ml_prob_velocity', 'ml_prob_acceleration',
             'rwp_delta_last_6', 'rwp_delta_last_12', 'ml_rwp_gap', 'ml_rwp_gap_delta_6',
             'runs_delta_6v6', 'wickets_delta_6', 'balls_to_death', 'entering_death', 'resource_efficiency']
print(df_valid[new_feats].isna().sum().to_string())

## 4. Correlation Analysis

Which features correlate most strongly with the 12-ball probability delta?

In [ ]:
# Define all candidate features for analysis
EXISTING_FEATURES = [
    'resource_win_prob', 'score_vs_par', 'pressure_index', 'dls_pressure_index',
    'run_rate_diff', 'current_run_rate', 'required_run_rate',
    'projected_score', 'projected_vs_venue_avg', 'expected_final_score',
    'resources_remaining', 'resource_pct', 'wickets_lost',
    'runs_last_12', 'runs_last_18', 'wickets_last_12', 'wickets_last_30',
    'boundary_pct_last_18', 'acceleration_potential',
    'team_strength_diff', 'batting_team_win_rate', 'bowling_team_win_rate',
    'situation_advantage', 'batting_pair_strength',
    'score_per_wicket', 'chase_difficulty',
    'is_powerplay', 'is_death_overs',
]

NEW_FEATURES = [
    'ml_prob',  # current ML probability — key context
    'ml_prob_delta_last_6', 'ml_prob_delta_last_12', 'ml_prob_velocity', 'ml_prob_acceleration',
    'rwp_delta_last_6', 'rwp_delta_last_12',
    'ml_rwp_gap', 'ml_rwp_gap_delta_6',  # ML-vs-physics divergence
    'runs_delta_6v6', 'wickets_delta_6',
    'balls_to_death', 'entering_death', 'resource_efficiency',
    'momentum_baseline',  # naive trend continuation — critical for comparison
]

ALL_FEATURES = EXISTING_FEATURES + NEW_FEATURES

# Drop rows with NaN in features
df_eda = df_valid[ALL_FEATURES + ['ml_delta_12', 'direction', 'residual_delta_12', 'direction_residual',
                                   'innings', 'phase', 'match_id']].dropna()
print(f'EDA dataset: {len(df_eda):,} rows ({len(df_eda)/len(df_valid)*100:.1f}% of valid)')

In [ ]:
# Spearman correlation with ml_delta_12 (PRIMARY target)
corr_spearman = df_eda[ALL_FEATURES + ['ml_delta_12']].corr(method='spearman')['ml_delta_12'].drop('ml_delta_12')
corr_spearman = corr_spearman.sort_values(key=abs, ascending=False)

# Also Pearson
corr_pearson = df_eda[ALL_FEATURES + ['ml_delta_12']].corr(method='pearson')['ml_delta_12'].drop('ml_delta_12')

# Combine into report
corr_report = pd.DataFrame({
    'Spearman': corr_spearman,
    'Pearson': corr_pearson.loc[corr_spearman.index],
    'Source': ['NEW' if f in NEW_FEATURES else 'Existing' for f in corr_spearman.index]
}).round(4)

print('=== Feature Correlations with ml_delta_12 (sorted by |Spearman|) ===')
print(corr_report.to_string())
print(f'\n⚠️ Key check: momentum_baseline correlation = {corr_report.loc["momentum_baseline", "Spearman"]:.4f}')
print('   (If this is dominant, ODM might just be learning trend continuation)')

In [ ]:
# Correlation heatmap — top 20 features
top20 = corr_spearman.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df_eda[top20 + ['ml_delta_12']].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Spearman Correlation — Top 20 Features vs ml_delta_12')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: correlation with ML delta, colored by new vs existing
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if f in NEW_FEATURES else '#3498db' for f in corr_spearman.index]
corr_spearman.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Spearman Correlation with ml_delta_12')
ax.set_title('Feature Correlation with ML Probability Delta (12 balls)')
ax.axvline(0, color='black', linewidth=0.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='NEW momentum/ML features'),
                    Patch(color='#3498db', label='Existing features')],
          loc='lower right')
plt.tight_layout()
plt.show()

## 5. Feature Importance — Quick XGBoost

Train a fast XGBoost on direction classification to get feature importances.
Uses leave-match-out split to avoid overfitting.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb

X = df_eda[ALL_FEATURES].values
y = df_eda['direction'].values  # ML delta direction (UP=1, DOWN=0)
groups = df_eda['match_id'].values

# 5-fold leave-match-out cross-validation
gkf = GroupKFold(n_splits=5)
importances_all = []
metrics = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    xgb_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        use_label_encoder=False
    )
    xgb_model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=False)
    
    y_pred = xgb_model.predict(X[val_idx])
    y_prob = xgb_model.predict_proba(X[val_idx])[:, 1]
    
    metrics.append({
        'fold': fold,
        'accuracy': accuracy_score(y[val_idx], y_pred),
        'f1': f1_score(y[val_idx], y_pred, average='macro'),
        'auc': roc_auc_score(y[val_idx], y_prob),
        'val_size': len(val_idx)
    })
    importances_all.append(xgb_model.feature_importances_)

metrics_df = pd.DataFrame(metrics)
print('=== XGBoost ML Direction Classification (5-fold Leave-Match-Out) ===')
print(metrics_df.to_string(index=False))
print(f'\nMean Accuracy: {metrics_df["accuracy"].mean():.4f} ± {metrics_df["accuracy"].std():.4f}')
print(f'Mean AUC:      {metrics_df["auc"].mean():.4f} ± {metrics_df["auc"].std():.4f}')
print(f'Mean F1:       {metrics_df["f1"].mean():.4f} ± {metrics_df["f1"].std():.4f}')

In [ ]:
# Average feature importance across folds
mean_importance = np.mean(importances_all, axis=0)
feat_imp = pd.Series(mean_importance, index=ALL_FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['#e74c3c' if f in NEW_FEATURES else '#3498db' for f in feat_imp.index]
feat_imp.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Mean Feature Importance (gain)')
ax.set_title('XGBoost Feature Importance — ML Direction Prediction')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='NEW features'),
                    Patch(color='#3498db', label='Existing features')],
          loc='lower right')
plt.tight_layout()
plt.show()

print('\nTop 15 features:')
for i, (feat, imp) in enumerate(feat_imp.head(15).items(), 1):
    tag = ' ★ NEW' if feat in NEW_FEATURES else ''
    print(f'  {i:2d}. {feat:30s} {imp:.4f}{tag}')

## 6. SHAP Analysis

Understand nonlinear effects and feature interactions.

In [ ]:
import shap

# Use last fold's model, sample for speed
sample_idx = np.random.RandomState(42).choice(len(X), size=min(10000, len(X)), replace=False)
X_sample = pd.DataFrame(X[sample_idx], columns=ALL_FEATURES)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)
print(f'SHAP values computed for {len(X_sample)} samples')

In [ ]:
# SHAP summary plot — beeswarm
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, max_display=25, show=False)
plt.title('SHAP Values — Direction Prediction')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP bar plot — mean absolute SHAP by feature
shap_imp = pd.Series(
    np.abs(shap_values).mean(axis=0), index=ALL_FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if f in NEW_FEATURES else '#3498db' for f in shap_imp.index]
shap_imp.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Feature Importance — Direction Prediction')
ax.legend(handles=[Patch(color='#e74c3c', label='NEW features'),
                    Patch(color='#3498db', label='Existing features')],
          loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP dependence plots for top 4 features
top4 = shap_imp.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for idx, feat in enumerate(top4):
    ax = axes[idx // 2][idx % 2]
    feat_idx = ALL_FEATURES.index(feat)
    shap.dependence_plot(feat_idx, shap_values, X_sample, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feat}')
plt.tight_layout()
plt.show()

## 7. Segment Analysis

Does feature importance change by innings and phase? 
This reveals whether we need separate models per segment.

In [ ]:
# Train separate XGBoost per innings and compare feature importance
segment_importances = {}

for innings in [1, 2]:
    mask = df_eda['innings'] == innings
    X_seg = df_eda.loc[mask, ALL_FEATURES].values
    y_seg = df_eda.loc[mask, 'direction'].values
    
    model_seg = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        use_label_encoder=False
    )
    model_seg.fit(X_seg, y_seg, verbose=False)
    segment_importances[f'Inn {innings}'] = model_seg.feature_importances_

for phase_name in ['Powerplay', 'Middle', 'Death']:
    mask = df_eda['phase'] == phase_name
    if mask.sum() < 1000:
        continue
    X_seg = df_eda.loc[mask, ALL_FEATURES].values
    y_seg = df_eda.loc[mask, 'direction'].values
    
    model_seg = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        use_label_encoder=False
    )
    model_seg.fit(X_seg, y_seg, verbose=False)
    segment_importances[phase_name] = model_seg.feature_importances_

seg_df = pd.DataFrame(segment_importances, index=ALL_FEATURES)
print('Top 10 features by segment:')
for seg in seg_df.columns:
    top10 = seg_df[seg].sort_values(ascending=False).head(10)
    print(f'\n--- {seg} ---')
    for feat, imp in top10.items():
        tag = ' ★' if feat in NEW_FEATURES else ''
        print(f'  {feat:30s} {imp:.4f}{tag}')

In [ ]:
# Heatmap: feature importance by segment
top15_overall = feat_imp.head(15).index.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(seg_df.loc[top15_overall], annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_title('Feature Importance by Segment (top 15 overall)')
plt.tight_layout()
plt.show()

## 8. Baseline Models — The Critical Comparison

Compare multiple baselines to understand signal strength:
1. **Naive** — always predict majority class
2. **Momentum** — if ML prob went up last 12 balls → predict UP ⚠️ **KEY BASELINE**
3. **Logistic Regression** — linear model on all features
4. **XGBoost** — full model (already trained above)

> **CRITICAL CHECK**: If XGBoost ≈ Momentum → ODM is just learning trivial trend continuation → no real edge.  
> XGBoost must meaningfully beat Momentum to justify building the model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Prepare data
X_all = df_eda[ALL_FEATURES].values
y_all = df_eda['direction'].values  # ML delta direction
groups_all = df_eda['match_id'].values
ml_prob_delta_12 = df_eda['ml_prob_delta_last_12'].values  # momentum signal

gkf = GroupKFold(n_splits=5)

naive_accs, mom_accs, lr_accs, xgb_accs = [], [], [], []
naive_aucs, mom_aucs, lr_aucs, xgb_aucs = [], [], [], []

for train_idx, val_idx in gkf.split(X_all, y_all, groups_all):
    y_val = y_all[val_idx]
    majority = int(y_all[train_idx].mean() >= 0.5)
    
    # 1. Naive: predict majority class
    naive_pred = np.full(len(val_idx), majority)
    naive_accs.append(accuracy_score(y_val, naive_pred))
    
    # 2. Momentum: if ML prob went UP last 12 balls → predict UP
    mom_pred = (ml_prob_delta_12[val_idx] > 0).astype(int)
    mom_accs.append(accuracy_score(y_val, mom_pred))
    mom_aucs.append(roc_auc_score(y_val, ml_prob_delta_12[val_idx]))
    
    # 3. Logistic Regression
    lr = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=1000, random_state=42))])
    lr.fit(X_all[train_idx], y_all[train_idx])
    lr_pred = lr.predict(X_all[val_idx])
    lr_prob = lr.predict_proba(X_all[val_idx])[:, 1]
    lr_accs.append(accuracy_score(y_val, lr_pred))
    lr_aucs.append(roc_auc_score(y_val, lr_prob))
    
    # 4. XGBoost
    xgb_m = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        use_label_encoder=False
    )
    xgb_m.fit(X_all[train_idx], y_all[train_idx], verbose=False)
    xgb_pred = xgb_m.predict(X_all[val_idx])
    xgb_prob = xgb_m.predict_proba(X_all[val_idx])[:, 1]
    xgb_accs.append(accuracy_score(y_val, xgb_pred))
    xgb_aucs.append(roc_auc_score(y_val, xgb_prob))

print('=== Baseline Comparison — ML Direction (5-fold Leave-Match-Out) ===')
print(f'{"Model":<30} {"Accuracy":>10} {"AUC":>10}')
print('-' * 52)
print(f'{"Naive (majority)":<30} {np.mean(naive_accs):>10.4f} {"N/A":>10}')
print(f'{"⚠️ Momentum (Δ12 > 0)":<30} {np.mean(mom_accs):>10.4f} {np.mean(mom_aucs):>10.4f}')
print(f'{"Logistic Regression":<30} {np.mean(lr_accs):>10.4f} {np.mean(lr_aucs):>10.4f}')
print(f'{"XGBoost":<30} {np.mean(xgb_accs):>10.4f} {np.mean(xgb_aucs):>10.4f}')

print(f'\n--- Critical Comparisons ---')
lift_vs_naive = np.mean(xgb_accs) - np.mean(naive_accs)
lift_vs_momentum = np.mean(xgb_accs) - np.mean(mom_accs)
print(f'XGBoost vs Naive:    {lift_vs_naive:+.4f} ({lift_vs_naive*100:+.1f}%)')
print(f'XGBoost vs Momentum: {lift_vs_momentum:+.4f} ({lift_vs_momentum*100:+.1f}%) ← THIS IS THE KEY NUMBER')
print(f'AUC lift vs Momentum: {np.mean(xgb_aucs) - np.mean(mom_aucs):+.4f}')
print()
if lift_vs_momentum > 0.03:
    print('✅ XGBoost meaningfully beats momentum → ODM has real signal beyond trend continuation')
elif lift_vs_momentum > 0.01:
    print('⚠️ Small lift over momentum → some signal, but ODM may partly be learning momentum')
else:
    print('❌ XGBoost ≈ Momentum → ODM is just learning trend continuation → need residual target')

## 9. Residual Target Analysis + Quantile Regression CI

**If XGBoost ≈ Momentum above**, the residual target becomes essential:
```
residual = ml_delta_12 - momentum_baseline
```
This removes trivial trend continuation → ODM must learn *unexpected* movement.

Also test quantile XGBoost for CI estimation on the ML delta.

In [ ]:
# --- Residual Direction Classification ---
y_residual = df_eda['direction_residual'].values

res_naive_accs, res_xgb_accs, res_xgb_aucs = [], [], []

for train_idx, val_idx in gkf.split(X_all, y_residual, groups_all):
    y_val = y_residual[val_idx]
    majority = int(y_residual[train_idx].mean() >= 0.5)
    res_naive_accs.append(accuracy_score(y_val, np.full(len(val_idx), majority)))
    
    xgb_r = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, use_label_encoder=False
    )
    xgb_r.fit(X_all[train_idx], y_residual[train_idx], verbose=False)
    xgb_pred_r = xgb_r.predict(X_all[val_idx])
    xgb_prob_r = xgb_r.predict_proba(X_all[val_idx])[:, 1]
    res_xgb_accs.append(accuracy_score(y_val, xgb_pred_r))
    res_xgb_aucs.append(roc_auc_score(y_val, xgb_prob_r))

print('=== Residual Direction (unexpected movement) ===')
print(f'Naive: {np.mean(res_naive_accs):.4f}')
print(f'XGBoost: {np.mean(res_xgb_accs):.4f} (AUC {np.mean(res_xgb_aucs):.4f})')
res_lift = np.mean(res_xgb_accs) - np.mean(res_naive_accs)
print(f'Lift: {res_lift:+.4f}')
if res_lift > 0.02:
    print('✅ Can predict unexpected movement → real alpha potential')
else:
    print('⚠️ Residual direction hard to predict → consider volatility model instead')

print('\n--- Quantile Regression: ML Delta CI ---')
y_delta = df_eda['ml_delta_12'].values

quantile_results = {}
for quantile, label in [(0.05, 'p5'), (0.5, 'p50'), (0.95, 'p95')]:
    preds_all = np.zeros(len(y_delta))
    for train_idx, val_idx in gkf.split(X_all, y_delta, groups_all):
        qr = xgb.XGBRegressor(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            objective='reg:quantileerror', quantile_alpha=quantile,
            subsample=0.8, colsample_bytree=0.8, random_state=42
        )
        qr.fit(X_all[train_idx], y_delta[train_idx], verbose=False)
        preds_all[val_idx] = qr.predict(X_all[val_idx])
    quantile_results[label] = preds_all

p5, p50, p95 = quantile_results['p5'], quantile_results['p50'], quantile_results['p95']
coverage = np.mean((y_delta >= p5) & (y_delta <= p95))
avg_width = np.mean(p95 - p5)
mae_median = np.mean(np.abs(y_delta - p50))

print(f'90% CI Coverage:   {coverage:.4f} (target: 0.90)')
print(f'Avg CI Width:      {avg_width:.4f}')
print(f'Median MAE:        {mae_median:.4f}')
print(f'Actual Std(delta): {np.std(y_delta):.4f}')
print(f'MAE < Std?         {"YES ✓" if mae_median < np.std(y_delta) else "NO ✗"}')

In [ ]:
# Visualize: predicted CIs vs actual deltas
sample = np.random.RandomState(42).choice(len(y_delta), 200, replace=False)
sample = np.sort(sample)

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(range(len(sample)), p5[sample], p95[sample], alpha=0.3, color='steelblue', label='90% CI')
ax.plot(range(len(sample)), p50[sample], color='steelblue', linewidth=1, label='Predicted median')
ax.scatter(range(len(sample)), y_delta[sample], color='red', s=8, alpha=0.5, label='Actual ML delta', zorder=5)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('Sample Ball Index')
ax.set_ylabel('ML Probability Delta (12 balls)')
ax.set_title(f'Predicted CIs vs Actual ML Prob Deltas (coverage={coverage:.1%})')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Conclusions & Decision Matrix

The key question is not just "can we predict direction?" but "can we predict it *beyond* simple momentum?"

In [ ]:
# Summary report
print('=' * 65)
print('ODDS DIRECTION MODEL — EDA SUMMARY (ML PROB TARGET)')
print('=' * 65)
print(f'\nData: {len(df_eda):,} balls from IPL + PSL')
print(f'Target: ML model probability delta over 12 balls')
print(f'Direction balance: UP={df_eda["direction"].mean():.1%} / DOWN={1-df_eda["direction"].mean():.1%}')

print(f'\n{"="*40}')
print(f'DIRECTION CLASSIFICATION')
print(f'{"="*40}')
print(f'Naive baseline:      {np.mean(naive_accs):.4f}')
print(f'Momentum baseline:   {np.mean(mom_accs):.4f} (AUC {np.mean(mom_aucs):.4f})')
print(f'Logistic Regression: {np.mean(lr_accs):.4f} (AUC {np.mean(lr_aucs):.4f})')
print(f'XGBoost:             {np.mean(xgb_accs):.4f} (AUC {np.mean(xgb_aucs):.4f})')
print(f'\nXGBoost vs Naive:    {lift_vs_naive:+.4f}')
print(f'XGBoost vs Momentum: {lift_vs_momentum:+.4f} ← KEY METRIC')

print(f'\n{"="*40}')
print(f'RESIDUAL (UNEXPECTED MOVEMENT)')
print(f'{"="*40}')
print(f'Naive:    {np.mean(res_naive_accs):.4f}')
print(f'XGBoost:  {np.mean(res_xgb_accs):.4f} (AUC {np.mean(res_xgb_aucs):.4f})')
print(f'Lift:     {res_lift:+.4f}')

print(f'\n{"="*40}')
print(f'CI ESTIMATION (Quantile XGBoost)')
print(f'{"="*40}')
print(f'90% Coverage: {coverage:.4f} (target: 0.90)')
print(f'CI Width:     {avg_width:.4f}')
print(f'Median MAE:   {mae_median:.4f} (vs std {np.std(y_delta):.4f})')

print(f'\n{"="*40}')
print(f'TOP FEATURES')
print(f'{"="*40}')
print('XGBoost importance:')
for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
    tag = ' ★ NEW' if feat in NEW_FEATURES else ''
    print(f'  {i}. {feat}: {imp:.4f}{tag}')
print('\nSHAP importance:')
for i, (feat, imp) in enumerate(shap_imp.head(5).items(), 1):
    tag = ' ★ NEW' if feat in NEW_FEATURES else ''
    print(f'  {i}. {feat}: {imp:.4f}{tag}')

print(f'\n{"="*65}')
print(f'DECISION MATRIX')
print(f'{"="*65}')
if lift_vs_momentum > 0.03:
    print('✅ STRONG signal beyond momentum → Build V1 ODM with ML delta target')
    print('   → Use XGBoost/quantile regression')
    print('   → Features: top 10 from SHAP analysis')
elif lift_vs_momentum > 0.01 and res_lift > 0.01:
    print('⚠️ Moderate signal → Build V1 with RESIDUAL target instead')
    print('   → Removes trivial momentum, captures unexpected movement')
    print('   → May need more data (record more live matches)')
elif lift_vs_momentum > 0.01:
    print('⚠️ Moderate signal, weak residual → Proceed cautiously')
    print('   → Try residual target + simpler model (logistic)')
    print('   → Consider: is this just learning ML internal patterns?')
else:
    print('❌ No signal beyond momentum → Pivot options:')
    print('   1. Volatility model (predict CI width, not direction)')
    print('   2. Regime detection (stable vs volatile phases)')
    print('   3. Wait for market odds data')
    print(f'   Momentum baseline already achieves {np.mean(mom_accs):.1%} accuracy')